# Project 2

#### Kyan and Rene

This will be a study on housing prices.

In [152]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier, export_graphviz, plot_tree
from sklearn.model_selection import learning_curve, GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn import tree
from IPython.display import display, HTML

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

# suppress futurewarnings only
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [153]:
sns.set_theme(context='notebook', style='whitegrid')

# set default plot size
plt.rcParams['figure.figsize'] = 6,4

In [154]:
display(HTML("<style>.container { width:90% !important; }</style>"))

In [155]:
input_file = "kc_house_data.csv"
df = pd.read_csv(input_file)

## Data Preprocessing

In [156]:
df.drop(['id', 'lat','long','yr_renovated', 'view', 'waterfront', 'zipcode', 'date'], axis=1, inplace=True)
df.dropna()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,condition,grade,sqft_above,sqft_basement,yr_built,sqft_living15,sqft_lot15
0,221900.0,3,1.00,1180,5650,1.0,3,7,1180,0.0,1955,1340,5650
1,538000.0,3,2.25,2570,7242,2.0,3,7,2170,400.0,1951,1690,7639
2,180000.0,2,1.00,770,10000,1.0,3,6,770,0.0,1933,2720,8062
3,604000.0,4,3.00,1960,5000,1.0,5,7,1050,910.0,1965,1360,5000
4,510000.0,3,2.00,1680,8080,1.0,3,8,1680,0.0,1987,1800,7503
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21592,360000.0,3,2.50,1530,1131,3.0,3,8,1530,0.0,2009,1530,1509
21593,400000.0,4,2.50,2310,5813,2.0,3,8,2310,0.0,2014,1830,7200
21594,402101.0,2,0.75,1020,1350,2.0,3,7,1020,0.0,2009,1020,2007
21595,400000.0,3,2.50,1600,2388,2.0,3,8,1600,0.0,2004,1410,1287


In [157]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21597 entries, 0 to 21596
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   price          21597 non-null  float64
 1   bedrooms       21597 non-null  int64  
 2   bathrooms      21597 non-null  float64
 3   sqft_living    21597 non-null  int64  
 4   sqft_lot       21597 non-null  int64  
 5   floors         21597 non-null  float64
 6   condition      21597 non-null  int64  
 7   grade          21597 non-null  int64  
 8   sqft_above     21597 non-null  int64  
 9   sqft_basement  21597 non-null  str    
 10  yr_built       21597 non-null  int64  
 11  sqft_living15  21597 non-null  int64  
 12  sqft_lot15     21597 non-null  int64  
dtypes: float64(3), int64(9), str(1)
memory usage: 2.1 MB


In [158]:
df.sample(10)

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,condition,grade,sqft_above,sqft_basement,yr_built,sqft_living15,sqft_lot15
6543,260000.0,4,2.25,2320,16800,2.0,4,9,2320,0.0,1977,2700,15680
9743,329000.0,3,2.00,1840,6755,1.0,3,8,1840,0.0,1989,1790,6459
7196,995000.0,4,3.50,2780,9550,2.0,5,10,2530,250.0,1978,2724,10634
19187,930000.0,3,3.25,2640,4080,2.0,3,9,1840,800.0,1912,1990,4080
17231,1450000.0,5,4.00,4070,11334,2.0,3,10,4070,0.0,2014,2640,9401
18099,240000.0,4,1.00,1910,16320,1.5,3,6,1910,0.0,1934,1380,9000
6034,282510.0,4,1.00,1450,32234,1.5,4,6,1450,0.0,1932,1530,10125
10586,278000.0,3,2.25,1590,9425,1.0,3,7,1170,420.0,1985,1590,9394
20541,258800.0,2,1.75,1290,1624,2.0,3,8,1290,0.0,2014,1410,1963
5142,235000.0,3,1.00,1050,7670,1.5,5,7,1050,0.0,1955,1220,7670


### Machine Learning

In [ ]:



predictors = ['sqft_living','grade', 'bathrooms', 'bedrooms', 'condition', 'sqft_lot', 'sqft_above']
target = 'price'
X = df[predictors].values
y = df[target].values

# split the data into training and testing sets
X_train, X_test, y_train, y_test = \
    train_test_split(X, y, test_size=0.3, random_state = 42)

# compute the most common price in the training set
most_common_price = pd.Series(y_train).value_counts().index[0]
print("Most common price in the training set:", most_common_price)

# scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# set up the grid search parameters
param_grid = {
    'n_neighbors': range(1, 22, 2),
    'p': [1, 2]
}
grid_search = GridSearchCV(KNeighborsRegressor(), param_grid, cv=10)
grid_search.fit(X_train, y_train)
print("Best parameters from grid search:", grid_search.best_params_)

# create regressor object
regr = KNeighborsRegressor(n_neighbors=19, p=1)
regr.fit(X_train, y_train)

# predicted prices based on the selected predictor variables
y_pred = regr.predict(X_test)

# display the first 10 predicted and actual prices
print("Predicted prices:", y_pred[:5])
print("Actual prices:", y_test[:5])

# compute the baseline rmse value
mean_target = y_train.mean()
mse_baseline = ((mean_target - y_train) ** 2).mean()
rmse_baseline = np.sqrt(mse_baseline)
print(f"Baseline RMSE: {rmse_baseline:.3f}")

# compute the baseline mae value
median_target = y_train.mean()
mae_baseline = (np.abs(median_target - y_train)).mean()
print(f"Baseline MAE: {mae_baseline:.3f}")

# compute the mean squared error
mse = np.mean((y_test - y_pred) ** 2)
print(f"Mean Squared Error: {mse:.3f}")

# compute the root mean squared error
rmse = np.sqrt(mse)
print(f"Root Mean Squared Error: {rmse:.3f}")

# compute the mean absolute error
mae = np.mean(np.abs(y_test - y_pred))
print(f"Mean Absolute Error: {mae:.3f}")



Most common price in the training set: 350000.0
Best parameters from grid search: {'n_neighbors': 19, 'p': 1}
Predicted prices: [214792.10526316 298660.21052632 363986.05263158 346739.47368421
 405651.05263158]
Actual prices: [132500. 415000. 494000. 355000. 606000.]
Baseline RMSE: 368717.154
Baseline MAE: 235842.040
Mean Squared Error: 51934038190.230
Root Mean Squared Error: 227890.408
Mean Absolute Error: 141250.656
